# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.

## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga y profiling

In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"

# Carga del archivo en un DataFrame
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
# Profiling: forma, tipos, nulos, duplicados y distribucion del target
print("Forma:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("\nDistribucion de target (conteo):")
print(df["target"].value_counts().sort_index())
print("\nDistribucion de target (proporcion):")
print(df["target"].value_counts(normalize=True).sort_index().round(3))

**Conclusión de calidad:**

> Describe brevemente qué encontraste y qué decisión tomaste.

## 2. Split train/test

In [ ]:
# Split estratificado 80/20 reproducible
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

In [ ]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

## 3. EDA sobre entrenamiento

Incluye:

- distribución de clases;
- distribuciones o boxplots de variables relevantes;
- correlaciones;
- tres hallazgos que puedan influir en el modelado.

In [ ]:
# EDA usando unicamente train_df
feature_cols = [c for c in train_df.columns if c != "target"]

# 1. Distribucion de clases
plt.figure()
sns.countplot(x="target", data=train_df)
plt.title("Distribucion de clases (train)")
plt.xlabel("Clase")
plt.ylabel("Conteo")
plt.show()

# 2. Boxplots de variables relevantes por clase
selected = feature_cols[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), selected):
    sns.boxplot(x="target", y=col, data=train_df, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# 3. Matriz de correlaciones
plt.figure(figsize=(10, 8))
sns.heatmap(train_df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=False)
plt.title("Correlaciones (train)")
plt.show()

# Resumen estadistico de las variables
train_df[feature_cols].describe().T

**Hallazgos:**

1.
2.
3.

## 4. Persistencia de datos y contrato

In [ ]:
# Reinicia indices y guarda train.csv y test.csv sin el indice
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

# Contrato de datos para las siguientes etapas
contract = {
    "target": "target",
    "features": [c for c in train_df.columns if c != "target"],
    "random_state": RANDOM_STATE,
    "test_size": 0.20,
}
(ARTIFACTS_DIR / "data_contract.json").write_text(
    json.dumps(contract, indent=2, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

- [ ] Cargué el CSV desde `data/raw`.
- [ ] Revisé calidad y distribución de la clase.
- [ ] Realicé un split estratificado.
- [ ] No exploré el conjunto de test.
- [ ] Guardé los tres artefactos requeridos.